## Author Name Clean

### 1  Imports and Path Configuration

In [ ]:
# ============================================================
# ✅ Imports and Path Configuration (Mac)
# ============================================================
import pandas as pd
import os
import re
import logging
from typing import Dict, List, Tuple, Optional, Set
from pathlib import Path

# Base directories
INTERMEDIATE = ""

# File paths
INPUT_FILE           = f"{INTERMEDIATE}/merged_full_2875_with_aff.csv"         # Input data (2875 papers)
CLEAN_FILE           = f"{INTERMEDIATE}/output_final_2875.csv"
OUTPUT_FILE          = f"{INTERMEDIATE}/output_final_standardized.csv"
COMPARE_FILE         = f"{INTERMEDIATE}/output_final_comparison.csv"

# Reference file for Chinese surnames
CHINESE_SURNAME_FILE = "/Users/urbana-lab/Desktop/名词补充/EPB引文网络/data/reference/Chinese_Family_Name.xlsx"

# Mapping and decision storage files
BASE_MAPPING_FILE    = f"{INTERMEDIATE}/base_mapping.json"
AI_DECISIONS_FILE    = f"{INTERMEDIATE}/ai_decisions.json"
AI_DECISIONS_PKL     = f"{INTERMEDIATE}/ai_decisions.pkl"
UPDATED_MAPPING_FILE = f"{INTERMEDIATE}/updated_mapping.json"
CONFLICTED_FILE      = f"{INTERMEDIATE}/conflicted_cases.pkl"
FINAL_MAPPING_FILE   = f"{INTERMEDIATE}/final_mapping.json"

# Setup logging
logging.basicConfig(level=logging.WARNING)
logger = logging.getLogger(__name__)

# ============================================================
# ✅ File Validation Checks
# ============================================================
print(f"📂 Current Working Directory: {os.getcwd()}\n")
print("File Checks:")

if os.path.exists(INPUT_FILE):
    size = os.path.getsize(INPUT_FILE) / 1024 / 1024
    print(f"  ✅ Input File: {INPUT_FILE} ({size:.1f} MB)")
else:
    print(f"  ❌ Input File Not Found: {INPUT_FILE}")

if os.path.exists(CHINESE_SURNAME_FILE):
    print(f"  ✅ Chinese Surname File: Found")
else:
    print(f"  ❌ Chinese Surname File Not Found: {CHINESE_SURNAME_FILE}")

print(f"\n✅ Cell 1 Complete")

## 2 Core Classes for Name Standardization

In [ ]:
# Core Classes: AuthorName, NameStandardizer, and splitting utilities
from dataclasses import dataclass
from collections import defaultdict

@dataclass
class AuthorName:
    """Data class to represent parsed author name components."""
    original: str
    last_name: str
    first_names: List[str]
    is_full_name: bool
    initials: List[str]

    def __hash__(self):
        return hash(self.original)
    def __eq__(self, other):
        return isinstance(other, AuthorName) and self.original == other.original


def split_author_string(author_string):
    """Splits a string of authors into a list based on common delimiters."""
    if not isinstance(author_string, str):
        return []
    if ';' in author_string:
        authors = [a.strip() for a in author_string.split(';')]
    elif '|' in author_string:
        authors = [a.strip() for a in author_string.split('|')]
    elif ' and ' in author_string:
        authors = [a.strip() for a in author_string.split(' and ')]
    elif ' & ' in author_string:
        authors = [a.strip() for a in author_string.split(' & ')]
    else:
        authors = [author_string.strip()]
    return [a for a in authors if a]


class NameStandardizer:
    """Class to handle parsing, matching, and grouping of author names."""
    def __init__(self):
        self.parsed_names = {}
        self.ambiguous_cases = []
        
        # Dictionary of common English name abbreviations and nicknames
        self.common_abbreviations = {
            'Michael': ['M', 'Mike', 'Mick'], 'Robert': ['R', 'Bob', 'Rob'],
            'William': ['W', 'Bill', 'Will'], 'James': ['J', 'Jim', 'Jimmy'],
            'David': ['D', 'Dave'], 'Richard': ['R', 'Rick', 'Dick'],
            'Thomas': ['T', 'Tom', 'Tommy'], 'Christopher': ['C', 'Chris'],
            'Daniel': ['D', 'Dan', 'Danny'], 'Matthew': ['M', 'Matt'],
            'Peter': ['P', 'Pete'], 'Jonathan': ['J', 'Jon'], 'Benjamin': ['B', 'Ben'],
        }
        self.chinese_surnames_pinyin = self._load_chinese_surnames()

    def parse_author_name(self, name_str):
        """Attempts to parse a raw name string into an AuthorName object."""
        if not isinstance(name_str, str):
            return None
        if name_str in self.parsed_names:
            return self.parsed_names[name_str]
        try:
            parsed_result = self._parse_name_flexible(name_str)
            if not parsed_result:
                return None
            last_name, first_names, _ = parsed_result
            if not first_names:
                return None
            
            # Check if any first name component is longer than a single initial
            is_full_name = any(len(n) > 1 for n in first_names)
            initials = [n[0].upper() for n in first_names if n]
            
            an = AuthorName(name_str, last_name, first_names, is_full_name, initials)
            self.parsed_names[name_str] = an
            return an
        except Exception as e:
            logger.warning(f"Failed to parse: {name_str}, {e}")
            return None

    def _parse_name_flexible(self, name_str):
        """Handles different naming formats (e.g., 'Last, First' vs 'First Last')."""
        name_str = name_str.strip()
        if ',' in name_str:
            parts = name_str.split(',', 1)
            last_name = parts[0].strip()
            first_part = parts[1].strip()
            comps = re.findall(r'[A-Za-z]+\.?', first_part)
            first_names = [c.replace('.', '').strip() for c in comps if c.strip()]
            return last_name, first_names, "Last,First"
        elif ' ' in name_str:
            return self._parse_first_last_format(name_str)
        else:
            return None

    def _parse_first_last_format(self, name_str):
        parts = name_str.split()
        if len(parts) < 2:
            return None
        if len(parts[0]) == 1: # Assuming initial first
            return ' '.join(parts[1:]), [parts[0]], "First Last"
        if len(parts) == 2:
            return parts[1], [parts[0]], "First Last"
        elif len(parts) >= 3:
            return parts[-1], parts[:-1], "First Middle Last"
        return None

    def _detect_name_order_swap(self, name1, name2):
        """Checks if First Name and Last Name were swapped."""
        if (name1.last_name.lower() in [n.lower() for n in name2.first_names] and
            name2.last_name.lower() in [n.lower() for n in name1.first_names]):
            a1 = set([name1.last_name.lower()] + [n.lower() for n in name1.first_names])
            a2 = set([name2.last_name.lower()] + [n.lower() for n in name2.first_names])
            if a1 == a2:
                return True
        return False

    def _are_different_chinese_names(self, name1, name2):
        """Special handling to differentiate Chinese names that might look similar as initials."""
        if not (self._is_chinese_surname(name1.last_name) and self._is_chinese_surname(name2.last_name)):
            return False
        if name1.last_name.lower() != name2.last_name.lower():
            return False
        if name1.is_full_name and name2.is_full_name:
            if (len(name1.first_names) == 1 and len(name2.first_names) == 1 and
                len(name1.first_names[0]) >= 2 and len(name2.first_names[0]) >= 2 and
                name1.first_names[0].lower() != name2.first_names[0].lower()):
                return True
        elif name1.is_full_name != name2.is_full_name:
            full = name1 if name1.is_full_name else name2
            init = name2 if name1.is_full_name else name1
            if len(full.first_names) == 1:
                ffn = full.first_names[0].lower()
                iparts = []
                for x in init.first_names:
                    p = x.replace(' ', '').replace('.', '')
                    if p:
                        iparts.extend(list(p.lower()))
                if len(iparts) >= 2:
                    return True
                elif len(iparts) == 1 and ffn[0] != iparts[0]:
                    return True
                else:
                    return False
        return False

    def _load_chinese_surnames(self):
        """Loads and converts Chinese surnames to Pinyin."""
        try:
            try:
                from pypinyin import lazy_pinyin, Style
                use_pypinyin = True
            except ImportError:
                use_pypinyin = False
                logger.warning("pypinyin library is not installed.")

            df = pd.read_excel(CHINESE_SURNAME_FILE)

            if use_pypinyin:
                surnames = df.iloc[:, 0].dropna().tolist()
                pinyin_set = set()
                for s in surnames:
                    if pd.isna(s) or not isinstance(s, str):
                        continue
                    s = str(s).strip()
                    if not s:
                        continue
                    py = ''.join(w.capitalize() for w in lazy_pinyin(s, style=Style.NORMAL))
                    pinyin_set.add(py)
                logger.warning(f"Converted {len(pinyin_set)} Chinese surnames using pypinyin.")
                return pinyin_set
            return set()
        except Exception as e:
            logger.error(f"Failed to load Chinese surnames: {e}")
            return set()

    def _is_chinese_surname(self, surname):
        return surname in self.chinese_surnames_pinyin

    def is_same_author_hard_rules(self, name1, name2):
        """Applies definitive rules to check if two author names match. Returns None if unsure."""
        if self._detect_name_order_swap(name1, name2):
            return True
        if name1.last_name.lower() != name2.last_name.lower():
            return False
        if self._are_different_chinese_names(name1, name2):
            return False
        if name1.original == name2.original:
            return True
        
        # Compare full names
        if name1.is_full_name and name2.is_full_name:
            if name1.first_names == name2.first_names:
                return True
            if name1.initials and name2.initials:
                if name1.initials[0].upper() != name2.initials[0].upper():
                    return False
            f1 = name1.first_names[0] if name1.first_names else ""
            f2 = name2.first_names[0] if name2.first_names else ""
            if len(f1) >= 2 and len(f2) >= 2 and f1.lower() != f2.lower():
                if self._could_be_english_nicknames(f1, f2):
                    return None
                return False
            if min(len(f1), len(f2)) == 1:
                return None
            return False
        
        # Compare full name with initials
        elif name1.is_full_name != name2.is_full_name:
            if name1.initials and name2.initials:
                if name1.initials[0].upper() != name2.initials[0].upper():
                    return False
                return None # Needs AI or manual review
            return None
        
        # Compare initials with initials
        elif not name1.is_full_name and not name2.is_full_name:
            if name1.first_names == name2.first_names:
                return True
            if self._is_initial_subset(name1, name2):
                return True
            return False
        return None

    def _is_initial_subset(self, name1, name2):
        """Checks if one set of initials is a valid subset of another (e.g., 'J' and 'J.R.')."""
        i1 = [n.upper() for n in name1.first_names if len(n) == 1]
        i2 = [n.upper() for n in name2.first_names if len(n) == 1]
        if len(i1) != len(name1.first_names) or len(i2) != len(name2.first_names):
            return False
        if len(i1) < len(i2) and i2[:len(i1)] == i1:
            return True
        elif len(i2) < len(i1) and i1[:len(i2)] == i2:
            return True
        return False

    def _could_be_english_nicknames(self, name1, name2):
        n1, n2 = name1.lower(), name2.lower()
        for full, nicks in self.common_abbreviations.items():
            fl = full.lower()
            nl = [n.lower() for n in nicks]
            if (n1 == fl and n2 in nl) or (n2 == fl and n1 in nl):
                return True
        return False

    def build_author_groups(self, author_names):
        """Groups author names based on hard rules, flags ambiguous cases."""
        parsed_names = []
        for s in author_names:
            p = self.parse_author_name(s)
            if p:
                parsed_names.append(p)
        logger.warning(f"Parsed {len(parsed_names)}/{len(author_names)} names.")

        # Group initially by last name to reduce search space
        by_lastname = defaultdict(list)
        for n in parsed_names:
            by_lastname[n.last_name.lower()].append(n)

        result = {}
        for lastname, names in by_lastname.items():
            full_names = [n for n in names if n.is_full_name]
            initials = [n for n in names if not n.is_full_name]
            groups = {}
            
            # Process full names
            for name in full_names:
                found = None
                for canon, grp in groups.items():
                    cp = self.parse_author_name(canon)
                    if cp and cp.first_names == name.first_names:
                        grp.add(name.original)
                        found = True
                        break
                if not found:
                    groups[name.original] = {name.original}
                    
            # Process initials against established full names
            for init in initials:
                matched = False
                for canon, grp in groups.items():
                    cp = self.parse_author_name(canon)
                    if cp and self.is_same_author_hard_rules(cp, init) is True:
                        grp.add(init.original)
                        matched = True
                        break
                if not matched:
                    found_i = None
                    for canon, grp in groups.items():
                        cp = self.parse_author_name(canon)
                        if cp and not cp.is_full_name and cp.first_names == init.first_names:
                            grp.add(init.original)
                            found_i = True
                            break
                    if not found_i:
                        groups[init.original] = {init.original}
            result.update(groups)

        # Flag ambiguous cases for AI resolution
        self.ambiguous_cases = []
        for i, n1 in enumerate(parsed_names):
            for n2 in parsed_names[i+1:]:
                if n1.last_name.lower() == n2.last_name.lower():
                    if self.is_same_author_hard_rules(n1, n2) is None:
                        self.ambiguous_cases.append((n1.original, n2.original))

        logger.warning(f"Created {len(result)} groups, {len(self.ambiguous_cases)} AI cases flagged.")
        return result

    def get_ai_assistance_cases(self):
        return self.ambiguous_cases

    def export_mapping(self, author_groups, filename):
        """Exports the author grouping to a JSON file."""
        import json
        mapping = {c: list(v) for c, v in author_groups.items()}
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(mapping, f, ensure_ascii=False, indent=2)
            
        total = sum(len(v) for v in mapping.values())
        multi = sum(1 for v in mapping.values() if len(v) > 1)
        
        print(f"\n=== Mapping Statistics ===")
        print(f"Total authors after standardization: {len(mapping):,}")
        print(f"Total original variants: {total:,}")
        print(f"Authors with multiple variants: {multi:,}")
        print(f"Compression rate: {((total - len(mapping)) / total * 100):.2f}%")
        return mapping

print("✅ Cell 2 Complete: Classes defined.")

## 3 Execute Standardization

In [ ]:
import time

# 1. Load data (using file with affiliations)
df = pd.read_csv(INPUT_FILE, encoding="utf-8-sig")
print(f"Total papers: {len(df)}")

# 2. Extract all authors (split and deduplicate)
all_authors = []
for authors_str in df['authors'].dropna():
    all_authors.extend(split_author_string(authors_str))

print(f"Total author occurrences: {len(all_authors):,}")

unique_authors = list(set(all_authors))
print(f"Unique author strings after deduplication: {len(unique_authors):,}")

# 3. Build groups
print(f"\nStarting grouping process (O(n²), please be patient)...")
t0 = time.time()

standardizer = NameStandardizer()
author_groups = standardizer.build_author_groups(unique_authors)

print(f"\nTime elapsed: {(time.time()-t0)/60:.1f} minutes")

# 4. Export base mapping and statistics
mapping = standardizer.export_mapping(author_groups, BASE_MAPPING_FILE)

# 5. Retrieve cases needing AI judgment
ai_cases = standardizer.get_ai_assistance_cases()
print(f"\nAmbiguous cases requiring AI judgment: {len(ai_cases):,}")
print(f"Base mapping saved to: {BASE_MAPPING_FILE}")

print(f"\n✅ Cell 4 Complete")

## 4: Test API Connection

In [ ]:
from openai import OpenAI

# WARNING: Hardcoded API key detected. Consider using environment variables for security.
client = OpenAI(
    api_key="key",
    base_url="url"
)

raw = client.chat.completions.with_raw_response.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": "Please reply: Connection successful"}],
)

print("HTTP Status Code:", raw.status_code)
print("Raw Response:")
print(raw.text)

## 5: Process AI Cases

In [ ]:
import json
import time

ai_cases = standardizer.get_ai_assistance_cases()
print(f"Starting processing of all {len(ai_cases)} cases...\n")

results = []  # Store all results

for i, (name1, name2) in enumerate(ai_cases, 1):
    # Judgment with retry logic
    result = None
    for attempt in range(3):  # Try up to 3 times
        try:
            # Assuming judge_same_author is defined elsewhere in your environment
            result = judge_same_author(name1, name2)
            break
        except Exception as e:
            print(f"  ⚠️ Error on case {i} (Attempt {attempt+1}): {e}")
            time.sleep(2)

    if result is None:
        result = "ERROR"  # Failed after 3 attempts, mark and continue

    results.append({
        "name1": name1,
        "name2": name2,
        "judgment": result
    })

    # Display progress (report every 20 cases)
    if i % 20 == 0:
        print(f"  Progress: {i}/{len(ai_cases)}")

    time.sleep(0.3)  # Rate limiting

print("\n✅ All processing complete!")

# ===== Save Results =====
output_path = "/Users/urbana-lab/Desktop/Preprocessing/data/intermediate/ai_judgments.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print(f"💾 Saved to: {output_path}")

# ===== Classification Statistics =====
from collections import Counter
counts = Counter(r["judgment"] for r in results)
print("\n=== Judgment Results Statistics ===")
for k, v in counts.items():
    print(f"  {k}: {v} cases")

## 6 Review Ambiguous Cases

In [ ]:
import json

with open("/Users/urbana-lab/Desktop/Preprocessing/data/intermediate/ai_judgments.json", encoding="utf-8") as f:
    results = json.load(f)

# Filter for cases that are not definitively "YES"
review = [r for r in results if r["judgment"] != "YES"]
print(f"Cases requiring manual review ({len(review)}):\n")

for i, r in enumerate(review, 1):
    print(f"{i:2d}. [{r['judgment']:6s}] 【{r['name1']}】 vs 【{r['name2']}】")

## 7 Merge, Normalize, and Save

In [ ]:
import json
import pandas as pd
from collections import defaultdict

# ===== 1. Load Data =====
with open("/Users/urbana-lab/Desktop/Preprocessing/data/intermediate/base_mapping.json", encoding="utf-8") as f:
    base_mapping = json.load(f)  # Format: {Standard Name: [List of Variants]}

with open("/Users/urbana-lab/Desktop/Preprocessing/data/intermediate/ai_judgments.json", encoding="utf-8") as f:
    ai_results = json.load(f)

# ===== 2. Reverse base_mapping for easy lookup -> {Variant: Standard Name} =====
variant_to_standard = {}
for standard, variants in base_mapping.items():
    for v in variants:
        variant_to_standard[v] = standard
    # Standard name maps to itself
    variant_to_standard[standard] = standard  

# ===== 3. Collect 'YES' pairs from AI and manual overrides =====
merge_pairs = [(r["name1"], r["name2"]) for r in ai_results if r["judgment"] == "YES"]
# Manual override added here
merge_pairs.append(("Wong, S C", "Wong, Sze-Chun")) 
print(f"Total pairs to merge: {len(merge_pairs)}")

# ===== 4. Union-Find (Disjoint Set) to process chaining relationships =====
# This algorithm efficiently groups authors A, B, and C together if A=B and B=C.
parent = {}
def find(x):
    parent.setdefault(x, x)
    while parent[x] != x:
        parent[x] = parent[parent[x]] # Path compression
        x = parent[x]
    return x

def union(a, b):
    parent[find(a)] = find(b)

# Build the sets
for n1, n2 in merge_pairs:
    union(n1, n2)

# ===== 5. Select the longest string (fullest name) as the standard for each group =====
groups = defaultdict(list)
for name in parent:
    groups[find(name)].append(name)

yes_mapping = {}
for root, members in groups.items():
    standard = max(members, key=len)  # Longest string generally equals full name
    for m in members:
        yes_mapping[m] = standard

# Force manual override for "Wong" to use full name
for m in ["Wong, S C", "Wong, Sze-Chun"]:
    if m in yes_mapping:
        yes_mapping[m] = "Wong, Sze-Chun"

# ===== 6. Final mapping function: Raw -> Base Standardization -> YES Merge =====
def final_name(raw):
    name = variant_to_standard.get(raw, raw)  # Step 1: Base standard mapping
    name = yes_mapping.get(name, name)        # Step 2: Apply AI/Manual 'YES' merges
    return name

# ===== 7. Apply to Dataset =====
df = pd.read_csv("/Users/urbana-lab/Desktop/Preprocessing/data/intermediate/merged_full_2875_with_aff.csv")

def normalize_authors(s):
    if pd.isna(s):
        return s
    authors = [a.strip() for a in str(s).split(";")]
    return "; ".join(final_name(a) for a in authors)

df["authors_normalized"] = df["authors"].apply(normalize_authors)

# ===== 8. Save Final Data =====
out_path = "/Users/urbana-lab/Desktop/Preprocessing/data/intermediate/merged_full_2875_with_aff_normalized.csv"
df.to_csv(out_path, index=False, encoding="utf-8-sig")
print(f"💾 Saved normalized dataset to: {out_path}")

# ===== 9. Final Statistics =====
raw_set, final_set = set(), set()
for s in df["authors"].dropna():
    for a in str(s).split(";"):
        raw_set.add(a.strip())
        
for s in df["authors_normalized"].dropna():
    for a in str(s).split(";"):
        final_set.add(a.strip())

print(f"\n=== Final Statistics ===")
print(f"Unique authors before standardization: {len(raw_set)}")
print(f"Unique authors after standardization: {len(final_set)}")
print(f"Total author identities merged/reduced: {len(raw_set) - len(final_set)}")